<a href="https://www.kaggle.com/code/pavankumar960/app-reviews-eda-sentiment-analysis?scriptVersionId=308393994" target="_blank"><img align="left" alt="Kaggle" title="Open in Kaggle" src="https://kaggle.com/static/images/open-in-kaggle.svg"></a>

# Project Setup

In [ ]:
# Basic Libs
import pandas as pd
import numpy as np

# Visualization Libs
import matplotlib.pyplot as plt
import seaborn as sns

#NLP Libs 
import nltk
from nltk.corpus import stopwords
from textblob import TextBlob

# Loading Dataset

In [ ]:
df=pd.read_csv("/kaggle/input/datasets/jahnavikachhia23/the-generative-ai-ecosystem-50k-user-reviews-2026/The_ Generative_AI_Ecosystem_50k_User_Reviews_2026.csv")

# Data Overview

In [ ]:
df.head()

In [ ]:
df.info()

In [ ]:
df.describe()

In [ ]:
df.nunique()

In [ ]:
App_uni = df['App'].unique()
print(App_uni)

In [ ]:
Rev_uni = df['Review_Theme'].unique()
print(Rev_uni)

In [ ]:
df.isnull().sum()

In [ ]:
total_duplicates = df.duplicated().sum()
print(f'Duplicates: {total_duplicates}')

In [ ]:
text_duplicates = df.duplicated(subset=['Review_Text']).sum()
print(f"Duplicate review texts: {text_duplicates}")

In [ ]:
duplicate_rows = df[df.duplicated(subset=['Review_Text'], keep=False)]

In [ ]:
print("\nHere are a few of the duplicate rows:")
print(duplicate_rows[['App', 'Review_Date', 'Review_Text']].head())

## **Observation**

* comparision in app between ['ChatGPT' 'Microsoft_Copilot' 'Google_Gemini' 'Perplexity' 'Claude']
* total 9 columns including target.
* Need to convert review_date column from Object to datetime.
* 5 columns are numerical.
* App_version makes sub category comparision
* App_version missing 6,912 values. Need to fill them as "Unknown".
* Review_Text as 9 duplicates.
* 'Star_Rating' and Sentiment_polarity is main evaluations.
* Review Theme gives new category to compare ['General' 'Accuracy/Logic Issues' 'Pricing/Subscription'
 'Bugs/Performance']
* thumbs_up_count weighs reviews.

# Data Cleaning & Preparation

## Handing App Version Null Values

In [ ]:
nulls_before = df['App_Version'].isnull().sum()
print(f"Nulls in App_Version before cleaning: {nulls_before}")

In [ ]:
df['App_Version'] = df['App_Version'].fillna('Unknown')

In [ ]:
nulls_after = df['App_Version'].isnull().sum()
print(f"Nulls in App_Version after cleaning: {nulls_after}")

## Handling convertion of Date dtype

In [ ]:
df['Review_Date'] = pd.to_datetime(df['Review_Date'], errors='coerce')

In [ ]:
df = df.sort_values(by='Review_Date')
df = df.drop_duplicates(subset=['App', 'Review_Text'], keep='last')

# Data Visualization

## Univariate Analysis

In [ ]:
numerical_cols = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_cols = df.select_dtypes(include=['object']).columns.tolist()
exclude_list = ['Review_Text', 'Review_Date', 'App_Version']
categorical_cols = [col for col in categorical_cols if col not in exclude_list]

print("Numerical Columns identified:", numerical_cols)
print("Categorical Columns identified:", categorical_cols)

In [ ]:
sns.set_theme(style="whitegrid")
    
fig, axes = plt.subplots(len(numerical_cols), 1, figsize=(8, 4 * len(numerical_cols)))
fig.tight_layout(pad=5.0)

for i, col in enumerate(numerical_cols):
    sns.histplot(df[col], kde=True, ax=axes[i], color='skyblue', bins=20)
    axes[i].set_title(f'Univariate Analysis: Distribution of {col}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Frequency')
    
plt.show()

### Univariate Numerical Observation
* 5 star ratings is upto 28k,1 star around 10k, 2 star around 2k, 3 start around 3k, 4 star around 5k reviews. it is left skew.
* words count btw 12 to 22 is around 26k, btw 22 to 27 is around 8k,4k,3k,2k,2k,2k,1k so until 100 words count. it is right skew.
* review length is right skew.
* thumbs up count is sharp right skew.
* sentiment is left skew but sharp rise at center(0).

In [ ]:
sns.set_theme(style="whitegrid")
fig, axes = plt.subplots(len(categorical_cols), 1, figsize=(8, 4 * len(categorical_cols)))
fig.tight_layout(pad=5.0)

for i, col in enumerate(categorical_cols):

    sns.countplot(
        data=df, 
        x=col, 
        ax=axes[i], 
        hue=col,
        legend=False,
        palette='Set2', 
        order=df[col].value_counts().index
    )
    axes[i].set_title(f'Univariate Analysis: Count of {col}', fontsize=14, fontweight='bold')
    axes[i].set_ylabel('Total Reviews')
    axes[i].tick_params(axis='x', rotation=15) 
    
plt.show()

In [ ]:
def analyze_thumbs_up_outliers(df):
    
    sns.set_theme(style="whitegrid")
    
    fig, axes = plt.subplots(3, 1, figsize=(10, 15))
    fig.tight_layout(pad=6.0)

    # Zoomed In View (0 to 100)
    zoomed_data = df[df['Thumbs_Up_Count'] <= 100]
    sns.histplot(zoomed_data['Thumbs_Up_Count'], bins=50, kde=True, color='skyblue', ax=axes[0])
    axes[0].set_title('1. Zoomed In: Thumbs Up (0-100 only)', fontsize=14, fontweight='bold')
    axes[0].set_ylabel('Number of Reviews')
    axes[0].set_xlabel('Thumbs Up Count')

    # Logarithmic Scale
    sns.histplot(df['Thumbs_Up_Count'] + 1, bins=50, color='purple', ax=axes[1], log_scale=True)
    axes[1].set_title('2. Log Scale: Viewing the entire range', fontsize=14, fontweight='bold')
    axes[1].set_ylabel('Number of Reviews')
    axes[1].set_xlabel('Thumbs Up Count (Log Scale: 1, 10, 100, 1000...)')

    # Boxplot
    sns.boxplot(x=df['Thumbs_Up_Count'], color='lightgreen', ax=axes[2])
    axes[2].set_title('3. Boxplot: Highlighting Extreme Outliers', fontsize=14, fontweight='bold')
    axes[2].set_xlabel('Thumbs Up Count')

    plt.show()

analyze_thumbs_up_outliers(df)

In [ ]:
import pandas as pd
import numpy as np

def handle_thumbs_up_outliers(df):
    """
    Isolates viral reviews and creates a capped version of the Thumbs_Up_Count.
    """
    # Isolate the "Viral" Reviews (Greater than 1000)
    viral_reviews = df[df['Thumbs_Up_Count'] >= 1000].sort_values(by='Thumbs_Up_Count', ascending=False)
    
    print(f"Found {len(viral_reviews)} Viral Reviews")
    print(viral_reviews[['App', 'Star_Rating', 'Thumbs_Up_Count', 'Review_Theme']].head(10))
    print("\n" + "="*50 + "\n")
    
    # Calculate the 99th Percentile
    p99 = df['Thumbs_Up_Count'].quantile(0.99)
    print(f"99% of all reviews have {p99:.0f} or fewer thumbs up.")
    
    # Cap the outliers (Winsorization)
    df['Thumbs_Up_Capped'] = df['Thumbs_Up_Count'].clip(upper=p99)
    
    print(f"Created new column 'Thumbs_Up_Capped' with a max value of {p99:.0f}.")
    
    return df
    
df = handle_thumbs_up_outliers(df)

## Bivariate Analysis

In [ ]:
def perform_bivariate_analysis(df):

    sns.set_theme(style="whitegrid")

    fig, axes = plt.subplots(1, 2, figsize=(16, 6))
    fig.tight_layout(pad=6.0)

    # Star Rating vs Sentiment Polarity
    sns.boxplot(
        data=df, 
        x='Star_Rating', 
        y='Sentiment_Polarity', 
        ax=axes[0], 
        hue='Star_Rating', 
        palette='coolwarm', 
        legend=False
    )
    axes[0].set_title('Sentiment Score by Star Rating', fontsize=14, fontweight='bold')
    axes[0].set_xlabel('Star Rating (1 to 5)')
    axes[0].set_ylabel('Sentiment Polarity (-1.0 to 1.0)')

    # App vs Average Capped Thumbs Up
    sns.barplot(
        data=df, 
        x='App', 
        y='Thumbs_Up_Capped', 
        ax=axes[1], 
        hue='App', 
        palette='viridis', 
        legend=False
    )
    axes[1].set_title('Average Thumbs Up per Review (Capped)', fontsize=14, fontweight='bold')
    axes[1].set_xlabel('App Name')
    axes[1].set_ylabel('Average Thumbs Up (Max 24)')
    axes[1].tick_params(axis='x', rotation=15)

    plt.show()

perform_bivariate_analysis(df)

### Bivariate Analysis Observation
* avg sentiment for 5 star is 0.7, for 4 star is 0.6, 0.25 for 3 star, 0 for 2 star, -0.2 for 1 star.
* microsoft have high 1.65, perplexity at 1, claude at 0.8, google at 0.4, chatgpt at 0.3 

## Multivariate Analysis

In [ ]:
def multivariate_analysis(df):

    sns.set_theme(style="whitegrid")

    plt.figure(figsize=(14, 7))

    sns.barplot(
        data=df,
        x='App',
        y='Star_Rating',
        hue='Review_Theme',
        palette='Set2'
    )
    
    plt.title('Average Star Rating by App and Review Theme', fontsize=16, fontweight='bold')
    plt.xlabel('App Name', fontsize=12)
    plt.ylabel('Average Star Rating', fontsize=12)
    
    plt.ylim(0, 5) 
    
    plt.legend(title='Review Theme', bbox_to_anchor=(1.05, 1), loc='upper left')
    
    plt.tight_layout()
    plt.show()

multivariate_analysis(df)

### Multivariate Analysis Observation
* copilot is around 4.2 as general, following claude and chatgpt at 4, gemini and perplexity at 3.6
* copilot is around 3.8 for pricing, following claude and chatgpt at 2.6, gemini 2.4, perplexity 2
* claude and copilot with highest accuracy at 3.2, following gemini and chatgpt at 2.2, perplexity at 2.2
* for bugs, copilot at 3.1, following chatgpt 2.8, claude 2.6, gemini 2.2 and perplexity at 2.1

# Final Conclusion and Executive Summary

## Project Overview
In this notebook, we performed a comprehensive Exploratory Data Analysis (EDA) on a dataset of 50,000 app reviews for five major AI platforms: ChatGPT, Microsoft Copilot, Google Gemini, Perplexity, and Claude. 

**Data Preprocessing Pipeline:**
* Handled 6,912 missing values in the `App_Version` column via imputation.
* Converted `Review_Date` to datetime objects for temporal accuracy.
* Executed smart deduplication, isolating and dropping 9 redundant entries while preserving genuine updated reviews.
* Addressed right-skewed extreme outliers in `Thumbs_Up_Count` by applying Winsorization (capping at the 99th percentile of 24) to maintain accurate visualizations.

## Key Findings & Business Intelligence

Through Univariate, Bivariate, and Multivariate analysis, several clear patterns emerged regarding user satisfaction and NLP sentiment scores:

1. **Overall Leader:** **Microsoft Copilot** dominates the dataset with the highest average baseline rating (~4.2) and the highest average positive sentiment polarity (1.65). It also drives the highest user engagement (thumbs-up count).
2. **The "Bugs" and "Pricing" Impact:** Complaints regarding "Bugs/Performance" and "Pricing/Subscription" universally drag down ratings. However, Copilot demonstrates the highest resilience, maintaining ~3.8 for pricing and ~3.1 for bugs, whereas competitors like Perplexity and Gemini drop into the low 2.0 range.
3. **Accuracy is Critical:** "Accuracy/Logic Issues" are the most damaging to user trust. Claude and Copilot handle this best (3.2), but the rest of the field struggles significantly (2.2).
4. **Sentiment Alignment:** The NLP sentiment scores perfectly align with user star ratings, scaling linearly from negative polarity at 1-star to highly positive polarity at 5-stars, validating the integrity of the dataset.